## Get Cookbook Context

In [ ]:
import os
import xml.etree.ElementTree as ET
import pandas as pd
import re

folder_path = "cookbook_textencoded"
data = []

for filename in os.listdir(folder_path):
    if not filename.endswith(".xml"):
        continue

    file_path = os.path.join(folder_path, filename)
    book_id = os.path.splitext(filename)[0]

    description_text = ""
    preface_text = ""

    try:
        tree = ET.parse(file_path)
        root = tree.getroot()

        # 读取 <meta> 下的 <dcDescription>
        meta = root.find("meta")
        if meta is not None:
            desc = meta.find("dcDescription")
            if desc is not None and desc.text:
                description_text = desc.text.strip()
            else:
                for elem in meta.iter():
                    if elem.tag is ET.Comment:
                        comment_text = elem.text or ""
                        match = re.search(r"<dcDescription>(.*?)</dcDescription>", comment_text, re.DOTALL)
                        if match:
                            description_text = match.group(1).strip()
                            break

        # read text in <div type="preface">
        for div in root.findall(".//div[@type='preface']"):
            texts = []
            for elem in div.iter():
                if elem.text:
                    texts.append(elem.text.strip())
            preface_text = " ".join(texts).strip()
            break 

        data.append({
            "book_id": book_id,
            "description": description_text,
            "preface": preface_text
        })

    except ET.ParseError as e:
        print(f"cannot parse {filename}: {e}")
        data.append({
            "book_id": book_id,
            "description": "",
            "preface": ""
        })

df = pd.DataFrame(data)
print(df.head())


  book_id                                        description  \
0    amem                                                      
1    amwh  Introduction. The Christian Family. A Christia...   
2    army  Manual For Army Cooks Prepared Under the Direc...   
3    aunt  Preface. Miscellaneous. Soups. Fish and Oyster...   
4    bart                                                      

                                             preface  
0                                                     
1                                                     
2                                                     
3  PREFACE. IN compiling these receipts, dear rea...  
4                                                     


In [18]:
df

,book_id,description,preface
0,amem,,
1,amwh,Introduction. The Christian Family. A Christia...,
2,army,Manual For Army Cooks Prepared Under the Direc...,
3,aunt,Preface. Miscellaneous. Soups. Fish and Oyster...,"PREFACE. IN compiling these receipts, dear rea..."
4,bart,,
...,...,...,...
71,wash,Beverages; Hings for Beauty and Hygiene; Bread...,PREFACE. A preface to a compilation of cooking...
72,whit,,PREFACE IN presenting this book of recipes to ...
73,wosu,Bread and Yeast. Breakfast and Tea Cakes. Eggs...,PREFACE. THIS little volume is sent out with a...
74,youn,Dignity of the Housekeeper; First Principles; ...,PREFACE. WHATEVER views may be suggested by th...


In [20]:
# save DataFrame to CSV
output_file = "cookbook_descriptions.csv"
df.to_csv(output_file, index=False, encoding='utf-8-sig')

## Preface text analysis

In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
import nltk

# 1. load data
df = pd.read_csv("non_american_books_with_descriptions.csv")
preface_list = df['preface'].dropna().astype(str).tolist()

# 2. set up simple stopwords
simple_stopwords = set("""
a about above after again against all am an and any are aren't as at be because been before being below between both
but by can't cannot could couldn't did didn't do does doesn't doing don't down during each few for from further had
hadn't has hasn't have haven't having he he'd he'll he's her here here's hers herself him himself his how how's i i'd
i'll i'm i've if in into is isn't it it's its itself let's me more most mustn't my myself no nor not of off on once
only or other ought our ours ourselves out over own same shan't she she'd she'll she's should shouldn't so some such
than that that's the their theirs them themselves then there there's these they they'd they'll they're they've this
those through to too under until up very was wasn't we we'd we'll we're we've were weren't what what's when when's where
where's which while who who's whom why why's with won't would wouldn't you you'd you'll you're you've your yours yourself
yourselves
""".split())

# 3. preprocess and lemmatize the preface text
lemmatizer = WordNetLemmatizer()

def preprocess_and_lemmatize(text):
    words = re.findall(r'\b[a-z]{2,}\b', text.lower())
    return ' '.join(lemmatizer.lemmatize(word) for word in words if word not in simple_stopwords)

preface_cleaned = [preprocess_and_lemmatize(p) for p in preface_list]

# 4. create TF-IDF matrix
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.95, min_df=2)
tfidf = vectorizer.fit_transform(preface_cleaned)
feature_names = vectorizer.get_feature_names_out()

# 5. train NMF model
n_topics = 10 # number of topics to extract
nmf_model = NMF(n_components=n_topics, random_state=42)
nmf_model.fit(tfidf)

# 6. get topics and their top words
def get_nmf_topics(model, feature_names, n_top_words=10):
    topics = []
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        topics.append((f"Theme {topic_idx + 1}", top_words))
    return topics

nmf_topics = get_nmf_topics(nmf_model, feature_names)

# 7. change topics to DataFrame
topics_df = pd.DataFrame([
    {'Theme': name, **{f'Word {i+1}': word for i, word in enumerate(words)}}
    for name, words in nmf_topics
])

topics_df

c:\Users\86173\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\decomposition\_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


,Theme,Word 1,Word 2,Word 3,Word 4,Word 5,Word 6,Word 7,Word 8,Word 9,Word 10
0,Theme 1,cook,dish,oriental,cookery,food,like,book,used,make,receipt
1,Theme 2,jewish,time,dietary,recipe,relation,information,study,mr,making,woman
2,Theme 3,street,mr,art,year,cooking,knowledge,country,experience,attention,chestnut
3,Theme 4,duty,young,household,mother,additional,regard,reader,chapter,housekeeper,receipt
4,Theme 5,true,given,described,character,section,recognized,mary,real,particular,really
5,Theme 6,america,foundation,published,publication,member,study,result,american,dish,palatable
6,Theme 7,wine,dinner,dish,guest,table,served,hot,according,fish,number
7,Theme 8,edition,book,hope,recipe,success,great,united,demand,month,december
8,Theme 9,roasting,mean,complete,wholesome,american,hot,expensive,russian,limited,boiling
9,Theme 10,friend,month,man,thought,day,feel,pleasure,author,good,recipe


In [ ]:
# 1. load data
df = pd.read_csv("american_books_with_descriptions.csv")
preface_list = df['preface'].dropna().astype(str).tolist()

# 2. set up simple stopwords
simple_stopwords = set("""
a about above after again against all am an and any are aren't as at be because been before being below between both
but by can't cannot could couldn't did didn't do does doesn't doing don't down during each few for from further had
hadn't has hasn't have haven't having he he'd he'll he's her here here's hers herself him himself his how how's i i'd
i'll i'm i've if in into is isn't it it's its itself let's me more most mustn't my myself no nor not of off on once
only or other ought our ours ourselves out over own same shan't she she'd she'll she's should shouldn't so some such
than that that's the their theirs them themselves then there there's these they they'd they'll they're they've this
those through to too under until up very was wasn't we we'd we'll we're we've were weren't what what's when when's where
where's which while who who's whom why why's with won't would wouldn't you you'd you'll you're you've your yours yourself
yourselves
""".split())

# 3. preprocess and lemmatize the preface text
lemmatizer = WordNetLemmatizer()

def preprocess_and_lemmatize(text):
    words = re.findall(r'\b[a-z]{2,}\b', text.lower())
    return ' '.join(lemmatizer.lemmatize(word) for word in words if word not in simple_stopwords)

preface_cleaned = [preprocess_and_lemmatize(p) for p in preface_list]

# 4. create TF-IDF matrix
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.95, min_df=2)
tfidf = vectorizer.fit_transform(preface_cleaned)
feature_names = vectorizer.get_feature_names_out()

# 5. train NMF model
n_topics = 10
nmf_model = NMF(n_components=n_topics, random_state=42)
nmf_model.fit(tfidf)

# 6. get topics and their top words
def get_nmf_topics(model, feature_names, n_top_words=10):
    topics = []
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        topics.append((f"Theme {topic_idx + 1}", top_words))
    return topics

nmf_topics = get_nmf_topics(nmf_model, feature_names)

# 7. change topics to DataFrame
topics_df = pd.DataFrame([
    {'Theme': name, **{f'Word {i+1}': word for i, word in enumerate(words)}}
    for name, words in nmf_topics
])

topics_df

c:\Users\86173\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\decomposition\_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


,Theme,Word 1,Word 2,Word 3,Word 4,Word 5,Word 6,Word 7,Word 8,Word 9,Word 10
0,Theme 1,book,cook,dish,school,recipe,art,best,year,cookery,place
1,Theme 2,new,receipt,feature,improvement,book,french,work,domestic,subject,additional
2,Theme 3,receipt,dish,similar,family,cake,department,proved,work,good,various
3,Theme 4,lady,edition,female,hope,book,solicited,march,success,contained,decided
4,Theme 5,mr,street,delmonico,meat,able,experience,art,complete,lady,knowledge
5,Theme 6,true,reason,italian,described,represent,recipe,real,housewife,thought,instance
6,Theme 7,study,time,food,pupil,eat,knowledge,physical,life,woman,help
7,Theme 8,health,climate,public,year,comfort,point,attention,country,american,art
8,Theme 9,contributor,object,housekeeper,contribution,domestic,practical,reliable,authority,original,fourth
9,Theme 10,note,foundation,society,series,uniform,cordial,staff,member,devoted,result
